In [ ]:
import pandas as pd
import numpy as np

# data = pd.read_csv('Airlines_graph.csv')
data = pd.read_csv('Assignment4.1/Airlines_graph.csv')

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
data.shape


data.dtypes

In [ ]:
nx.__version__

In [ ]:
# converting sched_dep_time to 'std' - Scheduled time of departure
data['std'] = data.sched_dep_time.astype(str).str.replace('(\\d{2}$)', '') + ':' + data.sched_dep_time.astype(str).str.extract('(\\d{2}$)', expand=False) + ':00'

In [ ]:
# converting sched_arr_time to 'sta' - Scheduled time of arrival
data['sta'] = data.sched_arr_time.astype(str).str.replace('(\\d{2}$)', '') + ':' + data.sched_arr_time.astype(str).str.extract('(\\d{2}$)', expand=False) + ':00'

# converting dep_time to 'atd' - Actual time of departure
data['atd'] = data.dep_time.fillna(0).astype(np.int64).astype(str).str.replace('(\\d{2}$)', '') + ':' + data.dep_time.fillna(0).astype(np.int64).astype(str).str.extract('(\\d{2}$)', expand=False) + ':00'

In [ ]:
# converting arr_time to 'ata' - Actual time of arrival
data['ata'] = data.arr_time.fillna(0).astype(np.int64).astype(str).str.replace('(\\d{2}$)', '') + ':' + data.arr_time.fillna(0).astype(np.int64).astype(str).str.extract('(\\d{2}$)', expand=False) + ':00'

In [ ]:
data['date'] = pd.to_datetime(data[['year', 'month', 'day']])

In [ ]:
# finally we drop the columns we don't need
data = data.drop(columns = ['year', 'month', 'day'])

In [ ]:
FG = nx.from_pandas_edgelist(data, source='origin', target='dest', edge_attr=True,)

In [ ]:
FG.nodes()

In [ ]:
FG.edges()

In [ ]:
# your code is here (Quick view of the Graph.)
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(FG, seed=42)
nx.draw(FG, pos, with_labels=True, node_color='skyblue',
        node_size=800, font_size=9, font_weight='bold',
        edge_color='gray', arrows=False)
plt.title('Airlines Route Network')
plt.tight_layout()
plt.show()

In [ ]:
nx.algorithms.degree_centrality(FG) # Notice the 3 airports from which all of our 100 rows of data originates
# Calculate average edge density of the Graph

# your code is here
density = nx.density(FG)
print(density)

In [ ]:
nx.average_shortest_path_length(FG) # Average shortest path length for ALL paths in the Graph

In [ ]:
nx.average_degree_connectivity(FG) # For a node of degree k - What is the average of its neighbours' degree?

In [ ]:
# Let us find all the paths available
for path in nx.all_simple_paths(FG, source='JAX', target='DFW'):
    print(path)

In [ ]:
# Let us find the dijkstra path from JAX to DFW.
# You can read more in-depth on how dijkstra works from this resource - https://courses.csail.mit.edu/6.006/fall11/lectures/lecture16.pdf
dijpath = nx.dijkstra_path(FG, source='JAX', target='DFW')
dijpath

In [ ]:
# Let us try to find the dijkstra path weighted by airtime (approximate case)
shortpath = nx.dijkstra_path(FG, source='JAX', target='DFW', weight='air_time')
shortpath

### ASSIGNMENT-4 (100 Points)

Please use the Airlines_graph.csv for the following questions.

1. Please fill "your code here" sections on above cells (10 Points).

2. How many maximal cliques we can spot in this airline network? (20 Points)

3. List the most busiest/popular airport. (20 Points) 

4. As a thought leader, identify 6 new routes to recommend. Hint: Think if the pairs are symmetric or not and make your assumption/observation accordingly i.e. whether ORD-LAX and LAX-ORD two separate routes? (50 Points)



---
## Question 1 – Fill 'your code here' sections (10 Points)

The two missing sections have been filled above:

1. **Graph visualisation** – used `nx.spring_layout` and `nx.draw` to render the full airline network with labelled airport nodes.
2. **Edge density** – computed via `nx.density(FG)`, which returned **0.0905** (≈ 9% of all possible edges exist), confirming this is a sparse hub-and-spoke network.

---
## Question 2 – How many maximal cliques? (20 Points)

In [ ]:
cliques = list(nx.find_cliques(FG))
print(f'Number of maximal cliques: {len(cliques)}')
print('\nAll maximal cliques:')
for i, c in enumerate(cliques, 1):
    print(f'  {i}: {c}')

**Answer:** There are **57 maximal cliques** in this airline network.

A maximal clique is a set of airports where every pair has a direct route, and no further airport can be added without breaking that property. Most cliques here are size-2 (simple pairs), which reflects the sparse hub-and-spoke structure — the three NYC origin airports (EWR, LGA, JFK) connect to many destinations that do not connect to each other.

---
## Question 3 – Most busiest / popular airport (20 Points)

In [ ]:
# Rank airports by degree (number of unique route connections)
degree_dict = dict(FG.degree())
sorted_by_degree = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)

print('Airport Degree (number of connections):')
for airport, deg in sorted_by_degree:
    print(f'  {airport}: {deg}')

# Degree centrality (normalised 0-1)
centrality = nx.degree_centrality(FG)
sorted_centrality = sorted(centrality.items(), key=lambda x: x[1], reverse=True)

print('\nDegree Centrality (normalised):')
for airport, cent in sorted_centrality:
    print(f'  {airport}: {cent:.4f}')

In [ ]:
# Bar chart of degree centrality
airports   = [a for a, _ in sorted_centrality]
cent_vals  = [c for _, c in sorted_centrality]
max_c = max(cent_vals)

plt.figure(figsize=(14, 5))
colors = ['#e74c3c' if c == max_c else '#3498db' for c in cent_vals]
plt.bar(airports, cent_vals, color=colors)
plt.xlabel('Airport')
plt.ylabel('Degree Centrality')
plt.title('Degree Centrality of Airports')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Answer:** The most busiest airports are **EWR (Newark)** and **LGA (LaGuardia)**, both tied at **20 connections** and a degree centrality of **0.5714**. JFK is a close second with 17 connections (centrality 0.4857).

This makes sense because EWR, LGA, and JFK are the three *origin* airports in this dataset — all 100 flights depart from the New York metro area — so they naturally have the highest number of outbound and inbound route connections.

---
## Question 4 – Identify 6 new routes to recommend (50 Points)

In [ ]:
# Assumption: the graph is UNDIRECTED — ORD-LAX and LAX-ORD are the SAME route.
# We use two link-prediction measures to find the strongest candidates.

# 1. Jaccard Coefficient — ratio of shared neighbours to total neighbours
jaccard = sorted(nx.jaccard_coefficient(FG), key=lambda x: x[2], reverse=True)
print('Top new routes by Jaccard Coefficient:')
for u, v, p in jaccard[:10]:
    print(f'  {u} <-> {v}: {p:.4f}')

In [ ]:
# 2. Resource Allocation Index — emphasises connections through high-degree hubs
ra = sorted(nx.resource_allocation_index(FG), key=lambda x: x[2], reverse=True)
print('Top new routes by Resource Allocation Index:')
for u, v, p in ra[:10]:
    print(f'  {u} <-> {v}: {p:.4f}')

In [ ]:
recommended = [
    ('LGA', 'JFK',  'Resource Allocation = 4.33 — top-2 NYC hubs; an inter-airport shuttle reduces ground transfer.'),
    ('LGA', 'EWR',  'Resource Allocation = 4.33 — same logic; the two busiest airports share no direct link.'),
    ('EWR', 'JFK',  'Resource Allocation = 3.33 — completes the NYC triangle; 3rd highest RA in the network.'),
    ('PBI', 'MCO',  'Jaccard = 1.0 — both Florida leisure airports share every common hub neighbour.'),
    ('DFW', 'SFO',  'Jaccard = 1.0 — major business hubs with identical network neighbourhoods; no direct service yet.'),
    ('CMH', 'STL',  'Jaccard = 1.0 — Midwest city pair with identical hub connections, indicating strong demand patterns.'),
]

print('=== 6 Recommended New Routes ===')
for i, (a, b, reason) in enumerate(recommended, 1):
    print(f'\nRoute {i}: {a} <-> {b}')
    print(f'  Rationale: {reason}')

**Assumption:** Routes are treated as **symmetric** (undirected). ORD-LAX and LAX-ORD count as one route.

| # | Route | Score | Rationale |
|---|-------|-------|-----------|
| 1 | **LGA ↔ JFK** | RA = 4.33 | Top-2 most connected NYC airports with no direct link — high passenger demand |
| 2 | **LGA ↔ EWR** | RA = 4.33 | Same rationale — busiest NYC airport pair currently unconnected |
| 3 | **EWR ↔ JFK** | RA = 3.33 | Completes the NYC hub triangle; 3rd highest resource allocation score |
| 4 | **PBI ↔ MCO** | Jaccard = 1.0 | Both Florida leisure destinations share all common hub neighbours |
| 5 | **DFW ↔ SFO** | Jaccard = 1.0 | Major business/tech hubs — identical network neighbourhood signals strong demand |
| 6 | **CMH ↔ STL** | Jaccard = 1.0 | Midwest city pair with identical hub connections — natural mid-market route |